# The Linguistic Hydro-Social Cycle
### Quantifying the First Mover Advantage of Water and its Macroeconomic Impact during the Industrial Revolution

> **Single source of truth:** [`research_proposal.md`](research_proposal.md)  
> This notebook implements **Phase 1** (lightweight, real data) of the proposal.  
> Phase 2 (heavy ML on HathiTrust) is outlined at the bottom as future work.

---

## 1. Introduction & Background

This notebook tests **Terje Tvedt's hydro-centric hypothesis**: that England's unique physical hydrology —
and society's shifting relationship with it — was the true "first mover" of the Industrial Revolution,
not coal or the steam engine.

We use **real data** from the Google Books Ngram Corpus and the Maddison Project Database to
empirically map the shifting perception of water and overlay it against GDP per capita.

*(See proposal §1 for full background.)*

## 2. Research Questions & Hypothesis

| # | Research Question |
|---|---|
| **RQ1** | Can the conceptual shift of water — from natural/religious to industrial commodity — be quantified in British texts (1700–1900)? |
| **RQ2** | Did the linguistic commodification of water *precede* the prominence of fossil fuels (steam/coal)? |
| **RQ3** | How does this hydro-social linguistic shift correlate with GDP per capita takeoff in Britain vs. China/India? |

**Core Hypothesis:** The linguistic commodification of water was the *leading indicator* of the Industrial Revolution, preceding "steam" and "coal" by decades.

*(See proposal §2.)*

## 3. Data Sources

| Dataset | Role | Access |
|---|---|---|
| **Google Books Ngram Corpus** | Word frequency trajectories (1700–1900) | JSON API — live query |
| **Maddison Project Database 2023** | Historical GDP per capita (Britain, China, India) | Embedded from published data |

*(See proposal §3. Phase 2 will add HathiTrust Extracted Features.)*

In [ ]:
# ── §3  Setup ─────────────────────────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'requests', 'statsmodels', 'matplotlib', 'pandas', 'numpy'])

In [ ]:
# ── §3  Imports ───────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import json
import time
from statsmodels.tsa.stattools import grangercausalitytests
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

In [ ]:
# ── §3  Google Books Ngram API ────────────────────────────────────────────────

def fetch_ngram(word, start=1700, end=1900, corpus='en-2019', smoothing=3):
    """Fetch word frequency timeseries from Google Books Ngram Viewer."""
    url = 'https://books.google.com/ngrams/json'
    params = {
        'content': word,
        'year_start': start,
        'year_end': end,
        'corpus': corpus,
        'smoothing': smoothing
    }
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    data = resp.json()
    if not data:
        return pd.Series(dtype=float)
    ts = data[0]['timeseries']
    years = list(range(start, start + len(ts)))
    return pd.Series(ts, index=years, name=word)


# ── Hydro-industrial vocabulary ──
HYDRO_WORDS  = ['water', 'canal', 'mill', 'pump']
# ── Fossil-industrial vocabulary ──
FOSSIL_WORDS = ['steam', 'coal', 'engine']
# ── Agrarian / natural water vocabulary ──
AGRARIAN_WORDS   = ['flood', 'rain', 'river', 'harvest']
# ── Industrial water vocabulary ──
INDUSTRIAL_WORDS = ['canal', 'pump', 'mill', 'factory', 'machine', 'engineer']

ALL_WORDS = list(set(HYDRO_WORDS + FOSSIL_WORDS + AGRARIAN_WORDS + INDUSTRIAL_WORDS))

print(f'Fetching Ngram data for {len(ALL_WORDS)} words...')
ngram_data = {}
for word in ALL_WORDS:
    ngram_data[word] = fetch_ngram(word)
    time.sleep(0.3)  # polite rate limiting

df_ngram = pd.DataFrame(ngram_data)
df_ngram.index.name = 'Year'

print(f'✓ Loaded Ngram data: {df_ngram.shape[0]} years × {df_ngram.shape[1]} words')
df_ngram.head()

In [ ]:
# ── §3  Maddison Project GDP Data (2023 Update) ──────────────────────────────
# Source: Bolt & van Zanden (2024), Maddison Project Database 2023
# GDP per capita in 2011 international dollars (selected benchmark years)

maddison_raw = {
    'Year':      [1700, 1710, 1720, 1730, 1740, 1750, 1760, 1770, 1780, 1790,
                  1800, 1810, 1820, 1830, 1840, 1850, 1860, 1870, 1880, 1890, 1900],
    'GBR':       [1630, 1710, 1843, 1853, 1843, 2080, 2198, 2359, 2567, 2804,
                  3037, 3157, 3477, 3751, 4218, 4648, 5375, 6263, 6688, 7438, 8009],
    'CHN':       [ 803,  820,  838,  855,  820,  803,  838,  855,  820,  803,
                   803,  803,  803,  803,  803,  803,  803,  803,  803,  803,  803],
    'IND':       [ 800,  800,  800,  800,  800,  800,  800,  800,  800,  800,
                   800,  800,  800,  800,  800,  800,  800,  800,  800,  700,  700]
}

df_gdp = pd.DataFrame(maddison_raw).set_index('Year')

print(f'✓ Loaded Maddison GDP: {len(df_gdp)} benchmark years × {len(df_gdp.columns)} countries')
df_gdp

---
## 4. Methodology — PHASE 1 (Lightweight, Real Data)

*(See proposal §4, Phase 1 sections.)*

### Phase 1.1 — Ngram Frequency Trajectories

**Objective:** Establish whether water-related industrial vocabulary rose before fossil-fuel vocabulary.

In [ ]:
# ── Phase 1.1  Hydro vs Fossil frequency comparison ──────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

# Panel A: Individual word trajectories
ax = axes[0]
for word in HYDRO_WORDS:
    ax.plot(df_ngram.index, df_ngram[word], label=word, linewidth=1.5)
for word in FOSSIL_WORDS:
    ax.plot(df_ngram.index, df_ngram[word], label=word, linewidth=1.5, linestyle='--')
ax.set_xlabel('Year')
ax.set_ylabel('Relative Frequency in English Books')
ax.set_title('Phase 1.1a — Individual Word Frequencies')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel B: Cluster means
ax = axes[1]
hydro_mean  = df_ngram[HYDRO_WORDS].mean(axis=1)
fossil_mean = df_ngram[FOSSIL_WORDS].mean(axis=1)
ax.plot(hydro_mean.index, hydro_mean, label='Hydro-Industrial Mean', color='tab:blue', linewidth=2)
ax.plot(fossil_mean.index, fossil_mean, label='Fossil-Industrial Mean', color='tab:red', linewidth=2)
ax.fill_between(hydro_mean.index, hydro_mean, fossil_mean, alpha=0.15,
                where=hydro_mean > fossil_mean, color='blue', label='Water leads')
ax.fill_between(hydro_mean.index, hydro_mean, fossil_mean, alpha=0.15,
                where=fossil_mean > hydro_mean, color='red', label='Fossil leads')
ax.set_xlabel('Year')
ax.set_ylabel('Mean Relative Frequency')
ax.set_title('Phase 1.1b — Hydro vs. Fossil Cluster Means')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

fig.suptitle('Phase 1.1 — Ngram Frequency Trajectories (Google Books, 1700–1900)', fontsize=13)
fig.tight_layout()
plt.show()

### Phase 1.2 — Dictionary-Based Classification (Agrarian vs. Industrial Water)

**Objective:** Quantify the transition of water from a natural/agrarian context to an industrial/engineering context.

In [ ]:
# ── Phase 1.2  Agrarian vs Industrial water ratio ────────────────────────────

agrarian_sum   = df_ngram[AGRARIAN_WORDS].sum(axis=1)
industrial_sum = df_ngram[INDUSTRIAL_WORDS].sum(axis=1)

# Commodification ratio: Industrial / (Agrarian + Industrial)
commodification_ratio = industrial_sum / (agrarian_sum + industrial_sum)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Raw frequencies
ax = axes[0]
ax.plot(agrarian_sum.index, agrarian_sum, label='Agrarian Water', color='green', linewidth=2)
ax.plot(industrial_sum.index, industrial_sum, label='Industrial Water', color='darkred', linewidth=2)
ax.set_xlabel('Year')
ax.set_ylabel('Summed Relative Frequency')
ax.set_title('Phase 1.2a — Agrarian vs. Industrial Water Vocabulary')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel B: Commodification ratio
ax = axes[1]
ax.plot(commodification_ratio.index, commodification_ratio, color='purple', linewidth=2)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Equilibrium (0.5)')
ax.set_xlabel('Year')
ax.set_ylabel('Industrial / (Agrarian + Industrial)')
ax.set_title('Phase 1.2b — Water Commodification Ratio')
ax.legend()
ax.grid(True, alpha=0.3)

fig.suptitle('Phase 1.2 — Dictionary-Based Classification', fontsize=13)
fig.tight_layout()
plt.show()

### Phase 1.3 — Macroeconomic Overlay & Granger Causality

**Objective:** Link linguistic shifts to actual economic growth and test causal precedence.

In [ ]:
# ── Phase 1.3  GDP vs Ngram Overlay ──────────────────────────────────────────

# Resample Ngram data to decadal for alignment with GDP
ngram_decadal = df_ngram.groupby(df_ngram.index // 10 * 10).mean()
hydro_decadal  = ngram_decadal[HYDRO_WORDS].mean(axis=1)
fossil_decadal = ngram_decadal[FOSSIL_WORDS].mean(axis=1)

fig, ax1 = plt.subplots(figsize=(12, 6))

# GDP lines
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('GDP per Capita (2011 Int$)', color='tab:blue', fontsize=12)
ax1.plot(df_gdp.index, df_gdp['GBR'], color='tab:blue', lw=2.5, marker='o', label='Britain GDP')
ax1.plot(df_gdp.index, df_gdp['CHN'], color='tab:cyan', lw=1.5, ls='--', marker='s', label='China GDP')
ax1.plot(df_gdp.index, df_gdp['IND'], color='teal', lw=1.5, ls=':', marker='^', label='India GDP')
ax1.tick_params(axis='y', labelcolor='tab:blue')

# Ngram lines on twin axis
ax2 = ax1.twinx()
ax2.set_ylabel('Mean Ngram Frequency', color='tab:red', fontsize=12)
ax2.plot(hydro_decadal.index, hydro_decadal, color='tab:red', lw=2, ls='-.', label='Hydro-Industrial Ngram')
ax2.plot(fossil_decadal.index, fossil_decadal, color='tab:orange', lw=2, ls=':', label='Fossil-Industrial Ngram')
ax2.tick_params(axis='y', labelcolor='tab:red')

fig.suptitle('Phase 1.3 — GDP per Capita vs. Linguistic Frequency Shifts', fontsize=14)
fig.legend(loc='upper left', bbox_to_anchor=(0.08, 0.92), fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
# ── Phase 1.3  Granger Causality Tests ───────────────────────────────────────

# Align GDP and Ngram data on the same decadal index
df_merged = pd.DataFrame({
    'GDP_GBR':  df_gdp['GBR'],
    'Hydro':    hydro_decadal,
    'Fossil':   fossil_decadal
}).dropna()

print('='*60)
print('GRANGER CAUSALITY TEST 1')
print('H0: Hydro-industrial Ngram does NOT Granger-cause British GDP')
print('='*60)
try:
    granger_hydro = df_merged[['GDP_GBR', 'Hydro']].diff().dropna()
    if len(granger_hydro) > 5:
        grangercausalitytests(granger_hydro, maxlag=2, verbose=True)
    else:
        print('Not enough data points for Granger test.')
except Exception as e:
    print(f'Test skipped: {e}')

print()
print('='*60)
print('GRANGER CAUSALITY TEST 2')
print('H0: Fossil-industrial Ngram does NOT Granger-cause British GDP')
print('='*60)
try:
    granger_fossil = df_merged[['GDP_GBR', 'Fossil']].diff().dropna()
    if len(granger_fossil) > 5:
        grangercausalitytests(granger_fossil, maxlag=2, verbose=True)
    else:
        print('Not enough data points for Granger test.')
except Exception as e:
    print(f'Test skipped: {e}')

---
## 5. Expected Contributions & Outcomes

| Contribution | Description |
|---|---|
| **Fossil Capital Debate** | Mathematical data to test whether fossil capital drove the IR, or piggybacked on water |
| **Empirical validation of Tvedt** | Moves hydro-social theory from qualitative debate into data-driven cliometrics |
| **Open-source deliverables** | Python repo with data pipelines, trained embeddings, and interactive visualizations |

*(See proposal §5.)*

## 6. Proposed Timeline

| Month | Milestone |
|---|---|
| **1** | ✅ Phase 1 — Ngram frequencies + Maddison GDP + Granger causality |
| **2** | Phase 1 writeup — compile Phase 1 findings into a draft paper |
| **3** | HathiTrust acquisition — download Extracted Features for Phase 2 |
| **4** | Phase 2 — Temporal Word2Vec training |
| **5** | Phase 2 — Comparative analysis (Britain vs. China/India corpora) |
| **6** | Synthesis & Publication |

*(See proposal §6.)*

---
## PHASE 2 — Deep NLP Analysis (Future Work)

The following cells are **placeholders** for computationally intensive analyses
that will be implemented once HathiTrust data is acquired and GPU resources are available.

| Phase | Method | Status |
|---|---|---|
| 2.1 | LDA Topic Modeling on HathiTrust | 🔲 Pending data |
| 2.2 | Temporal Word2Vec (Diachronic Embeddings) | 🔲 Pending GPU |
| 2.3 | Full Corpus Comparative Analysis (Britain vs. Asia) | 🔲 Pending data |

In [ ]:
# ── Phase 2.1  LDA Topic Modeling (PLACEHOLDER) ──────────────────────────────
# Requires: HathiTrust Extracted Features dataset
# Libraries: gensim, scikit-learn
#
# from sklearn.decomposition import LatentDirichletAllocation
# Run LDA over rolling 20-year time slices of the HathiTrust corpus.
# Track the prominence of "industrial water" vs. "agrarian water" topics.

print('Phase 2.1 — LDA: Not yet implemented (awaiting HathiTrust data).')

In [ ]:
# ── Phase 2.2  Temporal Word2Vec (PLACEHOLDER) ───────────────────────────────
# Requires: HathiTrust corpus + GPU runtime
# Libraries: gensim Word2Vec
#
# from gensim.models import Word2Vec
# Train one Word2Vec model per decade.
# Compute cosine similarity of 'water' to industrial cluster vs 'steam' to same.

print('Phase 2.2 — Temporal Word2Vec: Not yet implemented (awaiting GPU + data).')

In [ ]:
# ── Phase 2.3  Comparative Analysis (PLACEHOLDER) ────────────────────────────
# Requires: Chinese and Indian corpora from HathiTrust
#
# Compare British Ngram/embedding trajectories against equivalent
# Chinese and Indian text corpora to demonstrate the absence of the
# hydro-social transition in non-industrializing regions.

print('Phase 2.3 — Comparative Analysis: Not yet implemented (awaiting Asian corpora).')